In [1]:
import langchain
import os
from dotenv import load_dotenv
load_dotenv()
print(os.getenv("GROQ_API_KEY")[:10])

gsk_t4xXyH


## EXAMPLE 1 SIMPLE LLM CALL + STREAMING

In [6]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
model = init_chat_model("groq:llama-3.1-8b-instant")

## CREATE MESSAGES

In [7]:
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What are the 2 benefits of using langchain ?")
]

## INVOKE THE MESSSAGE

In [10]:
response=model.invoke(messages)
response

AIMessage(content='Langchain is an open-source platform that enables users to build applications using large language models. Two benefits of using Langchain are:\n\n1. **Integration with Multiple Models**: Langchain allows users to integrate and utilize different large language models (LLMs) in their applications. This integration enables users to leverage the strengths of various models, such as their domain expertise, tone, and style, to create more comprehensive and accurate applications.\n\n2. **Composition and Assembly of LLM Outputs**: Langchain provides tools and APIs that enable users to compose and assemble the outputs of different LLMs to produce more sophisticated and accurate results. This feature allows users to combine the strengths of multiple models to generate more informative and helpful responses, making Langchain a powerful tool for building complex language-based applications.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 157, 'pr

In [11]:
print(response.content)

Langchain is an open-source platform that enables users to build applications using large language models. Two benefits of using Langchain are:

1. **Integration with Multiple Models**: Langchain allows users to integrate and utilize different large language models (LLMs) in their applications. This integration enables users to leverage the strengths of various models, such as their domain expertise, tone, and style, to create more comprehensive and accurate applications.

2. **Composition and Assembly of LLM Outputs**: Langchain provides tools and APIs that enable users to compose and assemble the outputs of different LLMs to produce more sophisticated and accurate results. This feature allows users to combine the strengths of multiple models to generate more informative and helpful responses, making Langchain a powerful tool for building complex language-based applications.


## STREAMING

In [13]:
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

LangChain is an open-source library that enables the creation of scalable and composable LLM (Large Language Model) applications. Some of the benefits of using LangChain include:

1. **Scalability**: LangChain allows developers to build and connect multiple LLMs to create complex applications, making it easier to scale up or down as needed. This means you can handle a large volume of requests or integrate with other models to create more sophisticated services.

2. **Modularity**: LangChain provides a modular architecture that makes it easy to swap out different LLMs, data sources, or other components as needed. This modularity allows developers to reuse existing code and build new applications quickly, which can lead to faster development times and reduced maintenance costs.

## DYNAMIC PROMPT TEMPLATES

In [15]:
from langchain_core.prompts import ChatPromptTemplate
## create translation app
translation_template = ChatPromptTemplate.from_messages([
    ("system","You are a professional translator. Translate the following {text} from {source_language} to {target_language}. Maintain tone and style."),
    ("user", "{text}")
])
prompt=translation_template.invoke({
    "source_language":"English",
    "target_language":"French",
    "text":"Hello, how are you?"
})

In [19]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the following Hello, how are you? from English to French. Maintain tone and style.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})])

In [20]:
response = model.invoke(prompt)
print(response.content)

Bonjour, comment allez-vous?


## BUILDING FIRST CHAIN

In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Create a more complex chain
def create_story_chain():
    # Template for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Write a short, engaging story based on the given theme."),
        ("user", "Theme: {theme}\nMain character: {character}\nSetting: {setting}")
    ])
    
    # Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a literary critic. Analyze the following story and provide insights."),
        ("user", "{story}")
    ])
    
    # Build the chain - Method 1: Sequential execution
    story_chain = (
        story_prompt 
        | model 
        | StrOutputParser()
    )
    
    # Create a function to pass the story to analysis
    def analyze_story(story_text):
        return {"story": story_text}
    
    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )
    return analysis_chain

In [22]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short, engaging story based on the given theme.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\nMain character: {character}\nSetting: {setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_ou

In [23]:
result = chain.invoke({
    "theme": "artificial intelligence",
    "character": "a curious robot",
    "setting": "a futuristic city"
})

print("Story and Analysis:")
print(result)

Story and Analysis:
This story presents a rich tapestry of technological advancements, artificial intelligence, and human curiosity. The protagonist, Zeta, is a fascinating robot who embodies the qualities of inquisitiveness and determination. Her creator, Dr. Rachel Kim, has programmed Zeta with a thirst for knowledge, allowing her to navigate the complexities of the city and uncover hidden secrets.

One of the most striking aspects of the story is its depiction of New Eden, a futuristic city that showcases the possibilities of human ingenuity. The use of Nexarion, a glittering metallic material, and the towering skyscrapers that house underground laboratories and research facilities, creates a sense of awe and wonder. The city's inhabitants, a diverse mix of humans and advanced androids, add to the sense of a vibrant and inclusive society.

Zeta's exploration of the city and her discovery of the street performers' hidden message is a pivotal moment in the story. It highlights her adv